In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'
{'ID': "CONTRASEÑA_LZ_EN_SPARKY"}
# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-10 16:56:59 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



## Uso o activo

Un comercio usa adquirencia en un mes específico o mes de análisis cuando:

- Métrica Normal: tienen al menos 1 trx aporbada en ese mes.

- Métrica 5x: tiene al menos 1 trx aprobada en el transucurso de un año, incluyendo el mes de análisis


### Análisis ingestión compras tabla transaccional

In [3]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2026)
     AND MONTH IN (3, 4)
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_prueba = helper.obtener_dataframe(sql)

2026-04-09 08:54:42 - [INFO] - Transcurrido: 1775742882, Tiempo de Refresco = 1000


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 2/2 DATAFRAME        descargando   08:54:42 AM             

2026-04-09 08:54:50 - [INFO] - 965 filas, 10 columnas, 00:06.9 consultando, 00:00.5 descargando, 00:00.0 convirtiendo


 2/2 DATAFRAME         finalizado   08:54:42 AM     00:07.7 
------------------------------------------------------------


In [4]:
df_prueba.head(40)

,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2026,4,8,2026-04-07,1736284,1736284,1736284,1,1.0000,1.0000
1,2026,4,8,2026-04-06,39500,1781088,1781088,2,0.0222,1.0000
2,2026,4,7,2026-04-06,1741588,1781088,1741588,1,0.9778,0.9778
3,2026,4,8,2026-04-05,68,1911916,1911916,3,0.0000,1.0000
4,2026,4,7,2026-04-05,38388,1911916,1911848,2,0.0201,1.0000
5,2026,4,6,2026-04-05,1873460,1911916,1873460,1,0.9799,0.9799
6,2026,4,8,2026-04-04,10,2176359,2176359,3,0.0000,1.0000
7,2026,4,7,2026-04-04,236,2176359,2176349,2,0.0001,1.0000
8,2026,4,6,2026-04-04,2176113,2176359,2176113,1,0.9999,0.9999
9,2026,4,8,2026-04-03,11,1512843,1512843,3,0.0000,1.0000


## Construcción histórico transacciones

In [ ]:
# Verificar cantidad de registros por partición
# La última ingestión usada 2025-12-11 para obtener las transacciones. # MODIFICAR
sql = """
SELECT YEAR,
       mes,
       dia,
       count(*) AS frec
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2026
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3
ORDER BY YEAR DESC, mes DESC,
                    dia DESC;
"""
df_prueba = helper.obtener_dataframe(sql)


In [7]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
fecha_inicial = '2025-12-14' # MODIFICAR. DEBE SER EL PRIMER DÍA DE INGESTIÓN DE TRANSACCIONES A ALMACENAR O EL SIGUIENTE DÍA DESPUÉS DEL ÚLTIMO EN UNA ACTUALIZACIÓN.
fecha_final = '2026-04-08' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN

fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config

,fechas,year,month,day
0,2025-12-14,2025,12,14
1,2025-12-15,2025,12,15
2,2025-12-16,2025,12,16
3,2025-12-17,2025,12,17
4,2025-12-18,2025,12,18
...,...,...,...,...
111,2026-04-04,2026,4,4
112,2026-04-05,2026,4,5
113,2026-04-06,2026,4,6
114,2026-04-07,2026,4,7


In [ ]:
# # Crear tabla que almacenará la información

# sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_trxs;"""
# helper.ejecutar_consulta(sql_drop)

# sql = """
# CREATE TABLE proceso_vdm.mdo_adquirencia_trxs  (
#                 cod_unico VARCHAR,
#                 f_trx STRING,
#                 num_trxs BIGINT,
#                 mnt_total_trxs DECIMAL(38,2),
#                 DIA INT
#                 )
#             PARTITIONED BY 
#             (
#             YEAR INT,
#             MES INT
#             )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_trxs;"""

---------------------------------------------------------------------------------------
  i    tipo                 nombre                 estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------
 4/4      DROP proceso_vdm.mdo_adquirencia_trxs   finalizado   08:58:39 AM     00:00.9 
---------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------
  i    tipo                 nombre                 estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------
 5/5    CREATE proceso_vdm.mdo_adquirencia_trxs   finalizado   08:58:40 AM     00:00.7 
---------------------------------------------------------------------------------------


In [8]:
# Iterar para obtener las trxs por cliente
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month), '-', str(row.day))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    SELECT cod_unico,
       to_date(f_trx) as f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs,
        """ + str(row.day) + """ AS DIA,
        """ + str(row.year) + """ AS YEAR,
        """ + str(row.month) + """ AS MES
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = """ + str(row.year) + """
     AND MONTH = """ + str(row.month) + """
     AND DAY = """ + str(row.day) + """
     AND LOWER(TRIM(tipo_trx)) = "purchase"
     AND LOWER(TRIM(estado_trx)) = "cleared"
    GROUP BY 1,
            2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_trxs PARTITION (YEAR = """ + str(row.year) + """, MES = """ + str(row.month) + """)
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           DIA
    FROM proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

2026-04-09 14:10:04 - [INFO] - Transcurrido: 1775761805, Tiempo de Refresco = 1000


##################################################

Exrayendo datos de las particiones:  2025 - 12 - 14

Obteniendo transacciones adquirencia de los comercios

-----------------------------------------------------------------------------------
  i  tipo              nombre                  estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------
 1/1 DROP proceso.mdo_adquirencia_trxs_temp   finalizado   02:10:07 PM     00:01.3 
-----------------------------------------------------------------------------------
-------------------------------------------------------------------------------------
  i   tipo               nombre                  estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------
 2/2 CREATE proceso.mdo_adquirencia_trxs_temp   finalizado   02:10:08 PM     00:01.4 
----------------------------------------------------------------------------

In [11]:
# Verficar cantidad de registros desde la tabla fuente
sql = """
with outcome as (
SELECT cod_unico,
       to_date(f_trx) as f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = 2026
     AND MONTH = 4
     AND DAY = 7
     AND LOWER(TRIM(tipo_trx)) = "purchase"
     AND LOWER(TRIM(estado_trx)) = "cleared"
    GROUP BY 1,
            2
            )
SELECT count(*)
FROM outcome;
"""
helper.obtener_dataframe(sql)

--------------------------------------------------------------------------------------------
    i      tipo                 nombre                  estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 467/467 DATAFRAME                                    descargando   02:36:11 PM             

2026-04-09 14:36:15 - [INFO] - 1 filas, 1 columnas, 00:02.8 consultando, 00:00.8 descargando, 00:00.0 convirtiendo


 467/467 DATAFRAME                                     finalizado   02:36:11 PM     00:04.4 
--------------------------------------------------------------------------------------------


,count(*)
0,70244


In [12]:
# Verificar cantidad de registros por partición
# La última ingestión usada 2025-11-14 para obtener las transacciones. # MODIFICAR FECHA
sql = """
SELECT YEAR,
       mes,
       dia,
       count(*) AS frec
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2026
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3
ORDER BY YEAR DESC, mes DESC, dia DESC;
"""
helper.obtener_dataframe(sql).head(20)

--------------------------------------------------------------------------------------------
    i      tipo                 nombre                  estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 468/468 DATAFRAME                                    descargando   02:36:43 PM             

2026-04-09 14:36:52 - [INFO] - 1,044 filas, 4 columnas, 00:08.0 consultando, 00:00.8 descargando, 00:00.0 convirtiendo


 468/468 DATAFRAME                                     finalizado   02:36:43 PM     00:09.3 
--------------------------------------------------------------------------------------------


,year,mes,dia,frec
0,2026,4,8,70679
1,2026,4,7,70244
2,2026,4,6,256207
3,2026,4,1,74513
4,2026,3,31,72555
5,2026,3,30,186916
6,2026,3,27,72605
7,2026,3,26,70839
8,2026,3,25,68780
9,2026,3,24,233278


## Construcción histórico transacciones por mes

In [13]:

sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_trxs_mes_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_adquirencia_trxs_mes_1 STORED AS PARQUET AS
SELECT cod_unico,
       cast(replace(left(cast(f_trx AS string), 7), '-', '') AS INT) AS periodo_trxs,
       count(*) AS num_trxs,
       sum(mnt_total_trxs) AS mnt_total_trxs
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND """ + str(df_config.year.values[-1]) + """
AND MES BETWEEN 1 AND 12
GROUP BY 1,
       2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
    i      tipo                    nombre                    estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 469/469      DROP proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   02:37:30 PM     00:01.3 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
    i      tipo                    nombre                    estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 470/470    CREATE proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   02:37:31 PM     00:18.8 
-------------------------------------------------------------------------------------------------
--------------------

## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó adquirencia en el año 2024 [nuevo en 2024] y realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y siguientes no sumará a la métrica de nuevos.

In [2]:
# Tabla que almacenará el número de vinculaciones por mes
# Tabla que almacenará comercios con trxs por periodo
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist  (
                periodo DOUBLE,
                num_vinc BIGINT,
                tipo_cliente STRING
                )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist;"""
helper.ejecutar_consulta(sql_compute)

2026-04-10 16:57:09 - [INFO] - Transcurrido: 1775858230, Tiempo de Refresco = 1000


------------------------------------------------------------------------------------------
  i  tipo                  nombre                     estado     hora_inicio   duracion   
------------------------------------------------------------------------------------------
 1/1 DROP ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   04:57:11 PM     00:00.6 
------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
  i   tipo                   nombre                     estado     hora_inicio   duracion   
--------------------------------------------------------------------------------------------
 2/2 CREATE ..._adquirencia_1_a_1_vinculaciones_hist   finalizado   04:57:11 PM     00:00.6 
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

In [3]:
# Tabla que almacenará comercios con trxs por periodo
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist  (
                codigo_unico VARCHAR,
                periodo DOUBLE,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2),
                tipo_cliente STRING
                )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE INCREMENTAL STATS proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 4/4    DROP ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   04:57:14 PM     00:00.5 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 5/5  CREATE ...cia_1_a_1_vinculaciones_con_trxs_hist   finalizado   04:57:14 PM     00:00.7 
---------------------------------------------------------------------------------------------
------------------------------------------------------------

In [4]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2022-01-01' # MODIFICAR. INICIO DE UN MES. EN ACTUALIZACIÓN USAR EL SIGUIENTE MES DESPUÉS DEL ÚLTIMO ALMACENADO
fecha_final = '2026-03-31' # MODIFICAR. FIN DE UN MES. EN ACTUALIZACIÓN USAR EL MES RECIENTE CON TRANSACCIONES COMPLETAS
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['month_sgte_mes'] = df_config['periodo_sgte_mes'].dt.month
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)

# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11


In [5]:
# Leer archivo con el primer día de ingestión de cada mes de la tabla resultados_vspc_medios_de_pago.gsap_m_comercios
# Este archivo lo generá el archivo vinculacion_adquirencia.ipynb
df_primer_dia_ing_gsap_m_comercios = pd.read_excel('main_data/df_primer_dia_ing_vinc.xlsx')
df_primer_dia_ing_gsap_m_comercios.columns = ['year_sgte_mes', 'month_sgte_mes', 'primer_dia_ing_gsap_m_comercios', 'frec']
df_primer_dia_ing_gsap_m_comercios

,year_sgte_mes,month_sgte_mes,primer_dia_ing_gsap_m_comercios,frec
0,2022,1,1,456338
1,2022,2,1,460773
2,2022,3,1,466906
3,2022,4,1,474754
4,2022,5,2,481778
5,2022,6,1,488929
6,2022,7,1,496027
7,2022,8,1,502901
8,2022,9,1,510939
9,2022,10,3,519318


In [6]:
# Adicionar primer día de ingestión a df_config
df_config = df_config.merge(df_primer_dia_ing_gsap_m_comercios[['year_sgte_mes', 'month_sgte_mes', 'primer_dia_ing_gsap_m_comercios']], on=['year_sgte_mes', 'month_sgte_mes'], how='left')
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes,primer_dia_ing_gsap_m_comercios
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2,1
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3,1
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4,1
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5,2
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6,1
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7,1
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8,1
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9,1
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10,3
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11,1


In [7]:
df_config.head(20)

,fechas,year,month,day,periodo,periodo_base_year,periodo_viejos,periodo_sgte_mes,year_sgte_mes,month_sgte_mes,primer_dia_ing_gsap_m_comercios
0,2022-01-31,2022,1,31,202201,202201,202112,202202,2022,2,1
1,2022-02-28,2022,2,28,202202,202201,202112,202203,2022,3,1
2,2022-03-31,2022,3,31,202203,202201,202112,202204,2022,4,1
3,2022-04-30,2022,4,30,202204,202201,202112,202205,2022,5,2
4,2022-05-31,2022,5,31,202205,202201,202112,202206,2022,6,1
5,2022-06-30,2022,6,30,202206,202201,202112,202207,2022,7,1
6,2022-07-31,2022,7,31,202207,202201,202112,202208,2022,8,1
7,2022-08-31,2022,8,31,202208,202201,202112,202209,2022,9,1
8,2022-09-30,2022,9,30,202209,202201,202112,202210,2022,10,3
9,2022-10-31,2022,10,31,202210,202201,202112,202211,2022,11,1


In [8]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')

    print('Viejos')
    print('') 
    print('Obteniendo vinculaciones acumuladas al último mes del año anterior correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) <= """ + row.periodo_viejos + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados viejos en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'viejos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
       INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
       SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Nuevos')

    print('') 
    print('Obteniendo vinculaciones correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_1_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_1_new STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) = """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_1_new;"""
    helper.ejecutar_consulta(sql_compute)
    
    print('')
    print('Eliminar los nuevos que ya aparecen en los viejos, se puede presentar múltiples razones, entre ellas: adición de franquicias, la adición de una nueva franquicia genera un nuevo registro en la tabla')
    print('')
    print('Crear tabla con los viejos')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_viejos_temp_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_viejos_temp_new STORED AS PARQUET AS
    SELECT codigo_unico, periodo
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    WHERE tipo_cliente = 'viejos'"""
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_viejos_temp_new;"""
    helper.ejecutar_consulta(sql_compute)
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_new PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_new STORED AS PARQUET AS
    SELECT a.codigo_unico, a.periodo
    FROM proceso.mdo_adquirencia_vinculaciones_temp_1_new AS a
    LEFT ANTI JOIN proceso.mdo_adquirencia_vinculaciones_viejos_temp_new AS b on a.codigo_unico = b.codigo_unico
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_new;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados nuevos en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'nuevos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp_new
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('') 
    print('Obteniendo vinculaciones del año correspondiente, acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_1 PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_1 STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp_1;"""
    helper.ejecutar_consulta(sql_compute)
    
    print('')
    print('Eliminar los nuevos que ya aparecen en los viejos, se puede presentar múltiples razones, entre ellas: adición de franquicias, la adición de una nueva franquicia genera un nuevo registro en la tabla')
    print('')
    print('Crear tabla con los viejos')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_viejos_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_viejos_temp STORED AS PARQUET AS
    SELECT codigo_unico, periodo
    FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    WHERE tipo_cliente = 'viejos'"""
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_viejos_temp;"""
    helper.ejecutar_consulta(sql_compute)
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    SELECT a.codigo_unico, a.periodo
    FROM proceso.mdo_adquirencia_vinculaciones_temp_1 AS a
    LEFT ANTI JOIN proceso.mdo_adquirencia_vinculaciones_viejos_temp AS b on a.codigo_unico = b.codigo_unico
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')

    print('Todos')
    print('') 
    print('Obteniendo vinculaciones acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
      SELECT id_comercio as codigo_unico,
             """ + str(row.periodo) + """ AS periodo
      FROM resultados_vspc_medios_de_pago.gsap_m_comercios
      WHERE YEAR = """ + str(row.year_sgte_mes) + """
      AND MONTH = """ + str(row.month_sgte_mes) + """
      AND DAY = """ + str(row.primer_dia_ing_gsap_m_comercios) + """
      AND UPPER(estado_comercio) LIKE 'ACTIV%'
      AND CASE
               WHEN cast(id_comercio_padre AS BIGINT) = cast(nit AS BIGINT) THEN 1
               ELSE 0
            END = 1
      AND year(f_vinculacion_opy)*100 + month(f_vinculacion_opy) <= """ + str(row.periodo) + """;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando número de vinculados (todos) en proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist')
    print('')
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_hist
    SELECT periodo,
           count(*) AS num_vinc,
           'todos' as tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE INCREMENTAL STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON a.codigo_unico = b.cod_unico;
    """
    helper.ejecutar_consulta(sql)
    print('')

    

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2022 - 1

Viejos

Obteniendo vinculaciones acumuladas al último mes del año anterior correspondiente al Mes de análisis

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 7/7    DROP ...so.mdo_adquirencia_vinculaciones_temp   finalizado   04:58:09 PM     00:00.6 
---------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 8/8  CREATE ...so.mdo_adquirencia

In [9]:
# Obtener uso por mes
# Número de Vinculaciones
sql = """
SELECT tipo_cliente,
       periodo,
       count(*) AS num_vinc_uso_cumsum_ym
FROM proceso_vdm.mdo_adquirencia_1_a_1_vinculaciones_con_trxs_hist
GROUP BY 1,
         2
ORDER BY periodo DESC, tipo_cliente;
"""
df_prueba = helper.obtener_dataframe(sql)

2026-04-11 08:51:13 - [INFO] - Transcurrido: 51170, Tiempo de Refresco = 1000


-----------------------------------------------------------------------------------------------------
     i       tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------------
 1996/1996 DATAFRAME                                           descargando   08:51:13 AM             

2026-04-11 08:51:22 - [INFO] - 153 filas, 3 columnas, 00:08.4 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 1996/1996 DATAFRAME                                            finalizado   08:51:13 AM     00:08.9 
-----------------------------------------------------------------------------------------------------


In [10]:
df_prueba.head(50)

,tipo_cliente,periodo,num_vinc_uso_cumsum_ym
0,nuevos,202603.0,4602
1,todos,202603.0,110281
2,viejos,202603.0,105678
3,nuevos,202602.0,2617
4,todos,202602.0,106088
5,viejos,202602.0,103470
6,nuevos,202601.0,1004
7,todos,202601.0,99964
8,viejos,202601.0,98960
9,nuevos,202512.0,24455


In [11]:
df_prueba[df_prueba.tipo_cliente == 'todos']

,tipo_cliente,periodo,num_vinc_uso_cumsum_ym
1,todos,202603.0,110281
4,todos,202602.0,106088
7,todos,202601.0,99964
10,todos,202512.0,137535
13,todos,202511.0,134816
16,todos,202510.0,132463
19,todos,202509.0,129736
22,todos,202508.0,127041
25,todos,202507.0,124484
28,todos,202506.0,121329
